In [42]:
# Install dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [43]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"


In [44]:
# helper functions


def add_user_message(messages, text):  
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):  
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)




def chat(messages, system=None, temperature=1.0, stop_sequences=None):

  params = {
    "model": model,
    "max_tokens": 1000,
    "messages": messages,
    "temperature": temperature
  }
  
  if system:
    params["system"] = system
  if stop_sequences:
    params["stop_sequences"] = stop_sequences


  message = client.messages.create( **params )


  return message.content[0].text

In [45]:
from anthropic.types import ToolParam
# Tools and Schemas

from datetime import datetime, timedelta

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam( {
  
  "name": "get_current_datetime",
  "description": "Returns the current local date and time as a formatted string. Use this tool whenever you need to know the present date or time, such as for timestamping, scheduling, calculating durations relative to now, or answering questions about the current day, date, or time. The output is formatted using Python strftime-style directives.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A Python strftime format string that controls how the current datetime is rendered. Uses standard strftime directives, e.g. '%Y-%m-%d %H:%M:%S' produces '2025-06-19 14:30:00', '%Y-%m-%d' produces just the date, and '%H:%M' produces just the hour and minute. If omitted, defaults to '%Y-%m-%d %H:%M:%S'. Must be a non-empty string; passing an empty string will cause an error."
      }
    },
    "required": []
  }
})

In [46]:
messages = []

prompt = f"""
what is the current date and time?
"""    
add_user_message(messages, prompt)


response = client.messages.create(
  model=model,
  max_tokens=1000,
  messages=messages,
  tools=[get_current_datetime_schema],
)

messages.append({"role": "assistant", "content": response.content})

messages






[{'role': 'user', 'content': '\nwhat is the current date and time?\n'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01ETF75D8zzEAKHsH5x4Pc3F', caller=DirectCaller(type='direct'), input={}, name='get_current_datetime', type='tool_use')]}]

In [47]:
result = get_current_datetime(**response.content[0].input)
print(result)

2026-06-20 22:37:00


In [48]:
messages.append({
  "role": "user",
  "content": [
    {
      "type": "tool_result",
      "tool_use_id": response.content[0].id,
      "content": result,
      "is_error": False,
    }
  ]
})
messages

[{'role': 'user', 'content': '\nwhat is the current date and time?\n'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01ETF75D8zzEAKHsH5x4Pc3F', caller=DirectCaller(type='direct'), input={}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01ETF75D8zzEAKHsH5x4Pc3F',
    'content': '2026-06-20 22:37:00',
    'is_error': False}]}]

In [49]:
client.messages.create( 
  model=model,
  max_tokens=1000,
  messages=messages,
  tools=[get_current_datetime_schema],
)


Message(id='msg_01EM8X3mtaJZdnDGDECMBp4P', container=None, content=[TextBlock(citations=None, text='The current date and time is **June 20, 2026 at 10:37 PM** (22:37).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=813, output_tokens=31, output_tokens_details=None, server_tool_use=None, service_tier='standard'))